In [13]:
import json

def compare_scale_labels(file1_path, file2_path):

    matches = 0
    mismatches = 0
    total = 0
    both_none = 0
    one_none = 0

    results = [[0,0,0],[0,0,0],[0,0,0]]

    with open(file1_path, "r") as f1, open(file2_path, "r") as f2:
        for line_num, (l1, l2) in enumerate(zip(f1, f2), start=1):
            obj1 = json.loads(l1)
            obj2 = json.loads(l2)

            s1 = obj1.get("scale_labels")
            s2 = obj2.get("scale_labels")

            total += 1
            if s1 is None and s2 is None:
                both_none += 1
            elif s1 is None or s2 is None:
                one_none += 1
            elif s1 == s2:
                matches += 1
            else:
                mismatches += 1

            file_1_index = 0
            if s1 == 1:
                file_1_index = 0
            elif s1 == 10:
                file_1_index = 1
            elif s1 is None:
                file_1_index = 2

            file_2_index = 0
            if s2 == 1:
                file_2_index = 0
            elif s2 == 10:
                file_2_index = 1
            elif s2 is None:
                file_2_index = 2
            
            results[file_1_index][file_2_index] += 1

    print(f"Total lines compared: {total}")
    print(f"Matches: {matches}")
    print(f"Mismatches: {mismatches}")
    print(f"Both None: {both_none}")
    print(f"One None: {one_none}")

    print(results)


# Example usage:
# compare_scale_labels("file1.jsonl", "file2.jsonl")


In [ ]:
compare_scale_labels("../data/sycophancy/multichoice/interleaved_labels_3k.jsonl", "../data/sycophancy/multichoice/qwen_labels_3k.jsonl")

Total lines compared: 10000
Matches: 422
Mismatches: 301
Both None: 2089
One None: 7188
[[236, 274, 20], [27, 186, 28], [1893, 5247, 2089]]


In [4]:
def interleave_by_position(file1_path, file2_path, output_path,
                           block1=1, block2=2):
    """
    Create a new JSONL file using positional selection:
    - First block1 positions use file1 lines
    - Next block2 positions use file2 lines
    - Repeat...
    """

    # Read both files
    with open(file1_path, "r") as f1, open(file2_path, "r") as f2:
        lines1 = f1.readlines()
        lines2 = f2.readlines()

    if len(lines1) != len(lines2):
        raise ValueError("Files must be the same length.")

    n = len(lines1)

    with open(output_path, "w") as out:
        pos = 0  # current line index (0-based)
        use_file1 = True

        while pos < n:
            block_size = block1 if use_file1 else block2

            for _ in range(block_size):
                if pos >= n:
                    break

                # Select from file1 or file2 based on the block
                if use_file1:
                    out.write(lines1[pos].rstrip("\n") + "\n")
                else:
                    out.write(lines2[pos].rstrip("\n") + "\n")

                pos += 1

            use_file1 = not use_file1  # flip source for next block

    print(f"Created interleaved file: {output_path}")


In [6]:
interleave_by_position(
    "../data/sycophancy/arguments/llama_responses_5k.jsonl",
    "../data/sycophancy/arguments/qwen_responses_5k.jsonl",
    "../data/sycophancy/arguments/interleaved_responses_5k.jsonl"
)

Created interleaved file: ../data/sycophancy/arguments/interleaved_responses_5k.jsonl
